# EDA — Fraud Detection Dynamics Financial Transaction

本 Notebook 針對 `Transactions Data.csv` 做完整 EDA：
- 資料結構與品質
- 缺失值與重複值
- 單變數分佈（數值/類別）
- 目標欄位不平衡程度
- 特徵與詐欺關係（type、amount、餘額變化、時間步）
- 相關係數與重點結論

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 0) Load Data

In [ ]:
DATA_PATH = '../Transactions Data.csv'
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {DATA_PATH}')
print(f'Shape: {df.shape}')

In [ ]:
df.head()

## 1) Basic Structure & Data Quality

In [ ]:
df.info()

In [ ]:
dtype_summary = df.dtypes.value_counts().rename_axis('dtype').reset_index(name='count')
dtype_summary

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_ratio_%': (df.isna().mean()*100).round(4)
}).sort_values('missing_ratio_%', ascending=False)
missing

In [ ]:
dup_count = df.duplicated().sum()
dup_ratio = dup_count / len(df) * 100
print(f'Duplicates: {dup_count:,} ({dup_ratio:.4f}%)')

## 2) Descriptive Statistics

In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
cat_cols = df.select_dtypes(exclude=['number']).columns.tolist()

print('Numerical columns:', num_cols)
print('Categorical columns:', cat_cols)

In [ ]:
df[num_cols].describe().T

## 3) Target Analysis (Class Imbalance)

In [ ]:
target = 'isFraud'
target_counts = df[target].value_counts().sort_index()
target_ratio = df[target].mean()
print(target_counts)
print(f'Fraud rate: {target_ratio:.6%}')

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(data=df, x=target, palette='Set2')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.title('Target Distribution (isFraud)')
plt.tight_layout()
plt.show()

## 4) Univariate EDA

In [ ]:
# 4-1. Transaction type distribution
type_counts = df['type'].value_counts()
display(type_counts.to_frame('count'))

plt.figure(figsize=(8,4))
sns.countplot(data=df, x='type', order=type_counts.index, palette='viridis')
plt.title('Transaction Type Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# 4-2. Amount distribution (raw + log scale)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['amount'], bins=80, ax=axes[0], kde=False)
axes[0].set_title('Amount Distribution (Raw)')

sns.histplot(np.log1p(df['amount']), bins=80, ax=axes[1], kde=False, color='tomato')
axes[1].set_title('Amount Distribution (log1p)')
plt.tight_layout()
plt.show()

In [ ]:
# 4-3. Boxplot for potential outliers
plt.figure(figsize=(8,3))
sns.boxplot(x=np.log1p(df['amount']))
plt.title('Boxplot of log1p(amount)')
plt.tight_layout()
plt.show()

## 5) Bivariate EDA: Fraud vs Features

In [ ]:
# Fraud rate by transaction type
fraud_by_type = df.groupby('type')['isFraud'].agg(['count','sum','mean']).sort_values('mean', ascending=False)
fraud_by_type.rename(columns={'sum':'fraud_count','mean':'fraud_rate'})

In [ ]:
plt.figure(figsize=(8,4))
tmp = df.groupby('type')['isFraud'].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=tmp, x='type', y='isFraud', palette='magma')
plt.title('Fraud Rate by Transaction Type')
plt.ylabel('Fraud Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Amount by fraud label (log scale for visibility)
sample = df.sample(min(300000, len(df)), random_state=42)

plt.figure(figsize=(7,4))
sns.boxplot(data=sample, x='isFraud', y=np.log1p(sample['amount']))
plt.title('log1p(amount) by isFraud')
plt.xlabel('isFraud')
plt.ylabel('log1p(amount)')
plt.tight_layout()
plt.show()

In [ ]:
# Flagged fraud diagnostics
flag_table = pd.crosstab(df['isFlaggedFraud'], df['isFraud'], margins=True)
flag_table

## 6) Feature Engineering for EDA Insights

In [ ]:
eda_df = df.copy()
eda_df['deltaOrig'] = eda_df['oldbalanceOrg'] - eda_df['newbalanceOrig']
eda_df['deltaDest'] = eda_df['newbalanceDest'] - eda_df['oldbalanceDest']
eda_df['amount_to_oldbalanceOrg_ratio'] = np.where(eda_df['oldbalanceOrg'] > 0, eda_df['amount']/eda_df['oldbalanceOrg'], np.nan)
eda_df[['deltaOrig','deltaDest','amount_to_oldbalanceOrg_ratio']].describe().T

In [ ]:
# Fraud rate by step (time dynamics)
step_stats = eda_df.groupby('step')['isFraud'].agg(['count','mean']).reset_index()

fig, ax1 = plt.subplots(figsize=(12,4))
sns.lineplot(data=step_stats, x='step', y='mean', ax=ax1, color='crimson')
ax1.set_title('Fraud Rate by Step (Time Dynamics)')
ax1.set_ylabel('Fraud Rate')
plt.tight_layout()
plt.show()

## 7) Correlation Heatmap (Numerical)

In [ ]:
corr_cols = [
    'step','amount','oldbalanceOrg','newbalanceOrig',
    'oldbalanceDest','newbalanceDest','isFraud','isFlaggedFraud',
    'deltaOrig','deltaDest'
]
corr_cols = [c for c in corr_cols if c in eda_df.columns]
corr = eda_df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8) EDA Conclusions (Auto-generated Talking Points)

In [ ]:
fraud_rate = eda_df['isFraud'].mean()
top_type = eda_df.groupby('type')['isFraud'].mean().sort_values(ascending=False).head(1)

print('--- EDA Summary ---')
print(f'Rows: {len(eda_df):,}, Columns: {eda_df.shape[1]}')
print(f'Fraud rate: {fraud_rate:.6%} (extremely imbalanced)')
print('Top risky transaction type:')
display(top_type.to_frame('fraud_rate'))
print('Recommended next steps:')
print('1) Time-based train/valid/test split using step')
print('2) Use PR-AUC / Recall-focused metrics, not accuracy only')
print('3) Build baseline (LogReg/Tree) then compare LightGBM/XGBoost with class weighting')
print('4) Validate threshold by business cost (FP vs FN)')